In [ ]:
import duckdb
import hashlib
import pandas as pd
import numpy as np
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt

con = duckdb.connect()

con.execute("""CREATE OR REPLACE TABLE titanic AS SELECT * FROM read_csv_auto('/home/oleg/projects/personal_projects/dataanalysis_mcp/fixtures/titanic.csv', SAMPLE_SIZE=-1)""")
titanic_df = con.sql("SELECT * FROM titanic").df()

# --- Re-fit model 'titanic_logit' (kind=logistic) ---
expected_hash_titanic_logit = '7b9054553579934e061e3125097e4e473eabba3ec6b9e1afbc1d1ee684e462bc'
actual_hash_titanic_logit = hashlib.sha256(open('/home/oleg/projects/personal_projects/dataanalysis_mcp/fixtures/titanic.csv', 'rb').read()).hexdigest()
assert actual_hash_titanic_logit == expected_hash_titanic_logit, "Training data for 'titanic_logit' changed since the session was recorded."
titanic_logit = smf.logit("Survived ~ C(Sex) + C(Pclass) + Age + Fare + Q("Siblings/Spouses Aboard") + Q("Parents/Children Aboard")", data=titanic_df).fit(disp=False)

# --- Re-fit model 'sibsp_poisson' (kind=poisson) ---
expected_hash_sibsp_poisson = '7b9054553579934e061e3125097e4e473eabba3ec6b9e1afbc1d1ee684e462bc'
actual_hash_sibsp_poisson = hashlib.sha256(open('/home/oleg/projects/personal_projects/dataanalysis_mcp/fixtures/titanic.csv', 'rb').read()).hexdigest()
assert actual_hash_sibsp_poisson == expected_hash_sibsp_poisson, "Training data for 'sibsp_poisson' changed since the session was recorded."
sibsp_poisson = smf.poisson("Q("Siblings/Spouses Aboard") ~ C(Pclass) + C(Sex) + Age", data=titanic_df).fit(disp=False)

# --- Re-fit model 'sibsp_negbin' (kind=negbin) ---
expected_hash_sibsp_negbin = '7b9054553579934e061e3125097e4e473eabba3ec6b9e1afbc1d1ee684e462bc'
actual_hash_sibsp_negbin = hashlib.sha256(open('/home/oleg/projects/personal_projects/dataanalysis_mcp/fixtures/titanic.csv', 'rb').read()).hexdigest()
assert actual_hash_sibsp_negbin == expected_hash_sibsp_negbin, "Training data for 'sibsp_negbin' changed since the session was recorded."
sibsp_negbin = smf.negativebinomial("Q("Siblings/Spouses Aboard") ~ C(Pclass) + C(Sex) + Age", data=titanic_df).fit(disp=False)

# --- Re-fit model 'survival_logit' (kind=logistic) ---
expected_hash_survival_logit = '7b9054553579934e061e3125097e4e473eabba3ec6b9e1afbc1d1ee684e462bc'
actual_hash_survival_logit = hashlib.sha256(open('/home/oleg/projects/personal_projects/dataanalysis_mcp/fixtures/titanic.csv', 'rb').read()).hexdigest()
assert actual_hash_survival_logit == expected_hash_survival_logit, "Training data for 'survival_logit' changed since the session was recorded."
survival_logit = smf.logit("Survived ~ C(Sex) + C(Pclass) + Age + Q("Siblings/Spouses Aboard") + Q("Parents/Children Aboard") + Fare", data=titanic_df).fit(disp=False)

### Loaded dataset `titanic`
- Source: `/home/oleg/projects/personal_projects/dataanalysis_mcp/fixtures/titanic.csv`
- 887 rows x 8 columns

In [ ]:
con.execute("""
    CREATE OR REPLACE TABLE titanic AS
    SELECT * FROM read_csv_auto('/home/oleg/projects/personal_projects/dataanalysis_mcp/fixtures/titanic.csv', SAMPLE_SIZE=-1)
""")
titanic_df = con.sql("SELECT * FROM titanic").df()
titanic_df.head()

### Profiled `titanic`
- 887 rows x 8 columns
- Column `Sex` looks categorical — try compare_groups across it.
- Numeric column `Survived` is available — describe_column would surface outliers.

In [ ]:
profile_df = con.sql("SELECT * FROM titanic").df()
profile_df.describe(include='all')

### Missingness analysis — `titanic`
- No missingness detected
- 887 / 887 rows complete
- Top pattern: all 8 columns present (887 rows)
- Top suggestion: No missingness detected.

In [ ]:
raw_df = con.sql("SELECT * FROM titanic").df()
nulls = raw_df.isna()
print("Null percentages:")
print(nulls.mean().sort_values(ascending=False))

print("\nTop missingness patterns:")
print(nulls.groupby(list(nulls.columns)).size().sort_values(ascending=False).head(10))

# Pairwise phi via Pearson on 0/1:
print("\nPairwise missingness correlation:")
print(nulls.astype(int).corr().round(2))

### Compared `Survived` across `Sex` groups
- Test selected: **mann_whitney**
- statistic = 40319.5000, p_value = 0.0000
- effect_size (rank_biserial) = 0.5518
- Groups `male` and `female` differ significantly (Mann-Whitney U, p=0.0000, rank_biserial=0.552).

In [ ]:
from scipy import stats
df = con.sql('SELECT * FROM titanic').df()
# Test selected automatically by compare_groups: mann_whitney


### Compared `Survived` across `Pclass` groups
- Test selected: **kruskal_wallis**
- statistic = 101.1026, p_value = 0.0000
- effect_size (epsilon_squared) = 0.1141
- At least one of ['1', '2', '3'] differs significantly (Kruskal-Wallis, p=0.0000, epsilon_squared=0.114).

In [ ]:
from scipy import stats
df = con.sql('SELECT * FROM titanic').df()
# Test selected automatically by compare_groups: kruskal_wallis


### Compared `Age` across `Survived` groups
- Test selected: **mann_whitney**
- statistic = 96539.5000, p_value = 0.3677
- effect_size (rank_biserial) = -0.0359
- No statistically significant difference between `0` and `1` at α=0.05 (Mann-Whitney U, p=0.3677, rank_biserial=-0.036).

In [ ]:
from scipy import stats
df = con.sql('SELECT * FROM titanic').df()
# Test selected automatically by compare_groups: mann_whitney


### Compared `Fare` across `Survived` groups
- Test selected: **mann_whitney**
- statistic = 57578.0000, p_value = 0.0000
- effect_size (rank_biserial) = 0.3822
- Groups `0` and `1` differ significantly (Mann-Whitney U, p=0.0000, rank_biserial=0.382).

In [ ]:
from scipy import stats
df = con.sql('SELECT * FROM titanic').df()
# Test selected automatically by compare_groups: mann_whitney


### Correlation matrix on `titanic` (pearson)
- Columns: Survived, Pclass, Age, Siblings/Spouses Aboard, Parents/Children Aboard, Fare

In [ ]:
from scipy import stats
df = con.sql('SELECT "Survived", "Pclass", "Age", "Siblings/Spouses Aboard", "Parents/Children Aboard", "Fare" FROM titanic').df()
df.corr(method='pearson')

### Query

```
SELECT Sex, Pclass, COUNT(*) AS n, ROUND(AVG(Survived)*100, 1) AS survival_pct FROM titanic GROUP BY Sex, Pclass ORDER BY Sex, Pclass
```

- 6 rows returned

In [ ]:
con.sql('SELECT Sex, Pclass, COUNT(*) AS n, ROUND(AVG(Survived)*100, 1) AS survival_pct FROM titanic GROUP BY Sex, Pclass ORDER BY Sex, Pclass LIMIT 50').df()

### Query

```
SELECT CASE WHEN Age<13 THEN '00-12 child' WHEN Age<20 THEN '13-19 teen' WHEN Age<40 THEN '20-39 adult' WHEN Age<60 THEN '40-59 mid' ELSE '60+ senior' END AS age_band, COUNT(*) AS n, ROUND(AVG(Survived)*100,1) AS survival_pct FROM titanic GROUP BY age_band ORDER BY age_band
```

- 5 rows returned

In [ ]:
con.sql("SELECT CASE WHEN Age<13 THEN '00-12 child' WHEN Age<20 THEN '13-19 teen' WHEN Age<40 THEN '20-39 adult' WHEN Age<60 THEN '40-59 mid' ELSE '60+ senior' END AS age_band, COUNT(*) AS n, ROUND(AVG(Survived)*100,1) AS survival_pct FROM titanic GROUP BY age_band ORDER BY age_band LIMIT 50").df()

### Query

```
SELECT (CAST("Siblings/Spouses Aboard" AS INT) + CAST("Parents/Children Aboard" AS INT)) AS family_size, COUNT(*) AS n, ROUND(AVG(Survived)*100,1) AS survival_pct FROM titanic GROUP BY family_size ORDER BY family_size
```

- 9 rows returned

In [ ]:
con.sql('SELECT (CAST("Siblings/Spouses Aboard" AS INT) + CAST("Parents/Children Aboard" AS INT)) AS family_size, COUNT(*) AS n, ROUND(AVG(Survived)*100,1) AS survival_pct FROM titanic GROUP BY family_size ORDER BY family_size LIMIT 50').df()

### Fitted LOGISTIC model on `titanic`
- Formula: `Survived ~ C(Sex) + C(Pclass) + Age + Fare + Q("Siblings/Spouses Aboard") + Q("Parents/Children Aboard")`
- Strongest predictor: `C(Sex)[T.male]` (negative effect, statistically significant at α=0.05, p=5.895e-43). A one-unit increase changes the odds by ~-93.6% (odds ratio = 0.064).
- pseudo-R² = 0.3397
- AIC = 796.93, BIC = 835.23

In [ ]:
import statsmodels.formula.api as smf
df = con.sql("SELECT * FROM titanic").df()
model = smf.logit("Survived ~ C(Sex) + C(Pclass) + Age + Fare + Q("Siblings/Spouses Aboard") + Q("Parents/Children Aboard")", data=df).fit(disp=0)
model.summary()

### Evaluated `titanic_logit` on `titanic`
- ROC-AUC = 0.857, PR-AUC = 0.827, Brier = 0.139
- Accuracy = 0.803, F1 = 0.732 at threshold 0.5
- Calibration: 10 bins, max decile gap = 0.088

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, log_loss
y_true = titanic_df["Survived"]
y_pred = titanic_logit.predict(titanic_df)
print(f"ROC-AUC = {roc_auc_score(y_true, y_pred):.3f}")
print(f"PR-AUC  = {average_precision_score(y_true, y_pred):.3f}")
print(f"Brier   = {brier_score_loss(y_true, y_pred):.3f}")
# Calibration table (quantile bins):
_q = pd.qcut(y_pred, q=10, duplicates='drop', labels=False)
calibration = pd.DataFrame({'bin': _q + 1, 'y_true': y_true, 'y_pred': y_pred}).groupby('bin').agg(
    mean_predicted=('y_pred', 'mean'), mean_observed=('y_true', 'mean'), n=('y_true', 'size')
).reset_index()
calibration

### Adjusted 4 p-values via Benjamini–Hochberg (α=0.05)
- 3 / 4 hypotheses rejected after correction
- Largest adjusted-p still significant: 1.149e-21 (`Fare~Survived`)
- Not rejected: `Age~Survived`

In [ ]:
from statsmodels.stats.multitest import multipletests
p_raw = [1.3925485544345097e-58, 1.111328841715898e-22, 0.36773665395476707, 8.614270910587372e-22]
labels = ['Survived~Sex', 'Survived~Pclass', 'Age~Survived', 'Fare~Survived']
rejected, p_adj, _, _ = multipletests(p_raw, alpha=0.05, method='fdr_bh')
for lbl, raw, adj, rej in zip(labels, p_raw, p_adj, rejected):
    print(f"{lbl}  raw={raw:.4g}  adj={adj:.4g}  reject={rej}")

### Plotted `bar` of dataset `titanic`
- x: `Sex`
- y: `Survived`
- title: 'Survival Rate by Sex'

In [ ]:
import matplotlib.pyplot as plt
_df = con.sql("SELECT Sex, Survived FROM titanic").df()
fig, ax = plt.subplots(figsize=(8, 6), dpi=100)
_means = _df.groupby('Sex')['Survived'].mean()
ax.bar(_means.index.astype(str), _means.values)
ax.set_xlabel('Sex')
ax.set_ylabel('avg(Survived)')
ax.set_title('Survival Rate by Sex')
plt.show()

### Plotted `bar` of dataset `titanic`
- x: `Pclass`
- y: `Survived`
- title: 'Survival Rate by Class'

In [ ]:
import matplotlib.pyplot as plt
_df = con.sql("SELECT Pclass, Survived FROM titanic").df()
fig, ax = plt.subplots(figsize=(8, 6), dpi=100)
_means = _df.groupby('Pclass')['Survived'].mean()
ax.bar(_means.index.astype(str), _means.values)
ax.set_xlabel('Pclass')
ax.set_ylabel('avg(Survived)')
ax.set_title('Survival Rate by Class')
plt.show()

### Plotted `box` of dataset `titanic`
- x: `Survived`
- y: `Age`
- title: 'Age distribution by Survival'

In [ ]:
import matplotlib.pyplot as plt
_df = con.sql("SELECT Survived, Age FROM titanic").df()
fig, ax = plt.subplots(figsize=(8, 6), dpi=100)
_groups = _df.groupby('Survived')['Age'].apply(list)
ax.boxplot(_groups.values, tick_labels=_groups.index.astype(str).tolist())
ax.set_xlabel('Survived')
ax.set_ylabel('Age')
ax.set_title('Age distribution by Survival')
plt.show()

### Plotted `box` of dataset `titanic`
- x: `Pclass`
- y: `Fare`
- title: 'Fare by Class'

In [ ]:
import matplotlib.pyplot as plt
_df = con.sql("SELECT Pclass, Fare FROM titanic").df()
fig, ax = plt.subplots(figsize=(8, 6), dpi=100)
_groups = _df.groupby('Pclass')['Fare'].apply(list)
ax.boxplot(_groups.values, tick_labels=_groups.index.astype(str).tolist())
ax.set_xlabel('Pclass')
ax.set_ylabel('Fare')
ax.set_title('Fare by Class')
plt.show()

### Plotted `heatmap` of dataset `titanic`
- title: 'Numeric correlations'

In [ ]:
import matplotlib.pyplot as plt
_df = con.sql("SELECT * FROM titanic").df()
fig, ax = plt.subplots(figsize=(8, 6), dpi=100)
_corr = con.sql('SELECT * FROM titanic').df().corr(numeric_only=True)
im = ax.imshow(_corr.values, vmin=-1, vmax=1, cmap='RdBu_r')
ax.set_xticks(range(len(_corr.columns)))
ax.set_yticks(range(len(_corr.columns)))
ax.set_xticklabels(_corr.columns, rotation=45, ha='right')
ax.set_yticklabels(_corr.columns)
fig.colorbar(im, ax=ax)
ax.set_title('Numeric correlations')
plt.show()

### Fitted POISSON model on `titanic`
- Formula: `Q("Siblings/Spouses Aboard") ~ C(Pclass) + C(Sex) + Age`
- Strongest predictor: `Age` (negative effect, statistically significant at α=0.05, p=2.321e-37). A one-unit increase moves the response by -0.05111 units.
- pseudo-R² = 0.1146
- AIC = 1726.66, BIC = 1750.60
- Warnings: overdispersion

In [ ]:
import statsmodels.formula.api as smf
df = con.sql("SELECT * FROM titanic").df()
model = smf.poisson("Q("Siblings/Spouses Aboard") ~ C(Pclass) + C(Sex) + Age", data=df).fit(disp=0)
model.summary()

### Fitted NEGBIN model on `titanic`
- Formula: `Q("Siblings/Spouses Aboard") ~ C(Pclass) + C(Sex) + Age`
- Strongest predictor: `Age` (negative effect, statistically significant at α=0.05, p=9.416e-21). A one-unit increase multiplies the expected count by exp(β)=0.9537 (incidence rate ratio, IRR=0.9537).
- pseudo-R² = 0.0667
- AIC = 1622.00, BIC = 1650.73
- Warnings: underdispersion_vs_negbin

In [ ]:
import statsmodels.formula.api as smf
df = con.sql("SELECT * FROM titanic").df()
model = smf.negativebinomial("Q("Siblings/Spouses Aboard") ~ C(Pclass) + C(Sex) + Age", data=df).fit(disp=0)
model.summary()

### Loaded dataset `titanic`
- Source: `/home/oleg/projects/personal_projects/dataanalysis_mcp/fixtures/titanic.csv`
- 887 rows x 8 columns

In [ ]:
con.execute("""
    CREATE OR REPLACE TABLE titanic AS
    SELECT * FROM read_csv_auto('/home/oleg/projects/personal_projects/dataanalysis_mcp/fixtures/titanic.csv', SAMPLE_SIZE=-1)
""")
titanic_df = con.sql("SELECT * FROM titanic").df()
titanic_df.head()

### Profiled `titanic`
- 887 rows x 8 columns
- Column `Sex` looks categorical — try compare_groups across it.
- Numeric column `Survived` is available — describe_column would surface outliers.

In [ ]:
profile_df = con.sql("SELECT * FROM titanic").df()
profile_df.describe(include='all')

### Missingness analysis — `titanic`
- No missingness detected
- 887 / 887 rows complete
- Top pattern: all 8 columns present (887 rows)
- Top suggestion: No missingness detected.

In [ ]:
raw_df = con.sql("SELECT * FROM titanic").df()
nulls = raw_df.isna()
print("Null percentages:")
print(nulls.mean().sort_values(ascending=False))

print("\nTop missingness patterns:")
print(nulls.groupby(list(nulls.columns)).size().sort_values(ascending=False).head(10))

# Pairwise phi via Pearson on 0/1:
print("\nPairwise missingness correlation:")
print(nulls.astype(int).corr().round(2))

### Correlation matrix on `titanic` (pearson)
- Columns: Survived, Pclass, Age, Siblings/Spouses Aboard, Parents/Children Aboard, Fare

In [ ]:
from scipy import stats
df = con.sql('SELECT "Survived", "Pclass", "Age", "Siblings/Spouses Aboard", "Parents/Children Aboard", "Fare" FROM titanic').df()
df.corr(method='pearson')

### Compared `Survived` across `Sex` groups
- Test selected: **mann_whitney**
- statistic = 40319.5000, p_value = 0.0000
- effect_size (rank_biserial) = 0.5518
- Groups `male` and `female` differ significantly (Mann-Whitney U, p=0.0000, rank_biserial=0.552).

In [ ]:
from scipy import stats
df = con.sql('SELECT * FROM titanic').df()
# Test selected automatically by compare_groups: mann_whitney


### Compared `Survived` across `Pclass` groups
- Test selected: **kruskal_wallis**
- statistic = 101.1026, p_value = 0.0000
- effect_size (epsilon_squared) = 0.1141
- At least one of ['1', '2', '3'] differs significantly (Kruskal-Wallis, p=0.0000, epsilon_squared=0.114).

In [ ]:
from scipy import stats
df = con.sql('SELECT * FROM titanic').df()
# Test selected automatically by compare_groups: kruskal_wallis


### Compared `Age` across `Survived` groups
- Test selected: **mann_whitney**
- statistic = 96539.5000, p_value = 0.3677
- effect_size (rank_biserial) = -0.0359
- No statistically significant difference between `0` and `1` at α=0.05 (Mann-Whitney U, p=0.3677, rank_biserial=-0.036).

In [ ]:
from scipy import stats
df = con.sql('SELECT * FROM titanic').df()
# Test selected automatically by compare_groups: mann_whitney


### Compared `Fare` across `Survived` groups
- Test selected: **mann_whitney**
- statistic = 57578.0000, p_value = 0.0000
- effect_size (rank_biserial) = 0.3822
- Groups `0` and `1` differ significantly (Mann-Whitney U, p=0.0000, rank_biserial=0.382).

In [ ]:
from scipy import stats
df = con.sql('SELECT * FROM titanic').df()
# Test selected automatically by compare_groups: mann_whitney


### Query

```
SELECT Sex, Pclass, COUNT(*) AS n, ROUND(AVG(Survived)*100, 1) AS survival_pct FROM titanic GROUP BY Sex, Pclass ORDER BY Sex, Pclass
```

- 6 rows returned

In [ ]:
con.sql('SELECT Sex, Pclass, COUNT(*) AS n, ROUND(AVG(Survived)*100, 1) AS survival_pct FROM titanic GROUP BY Sex, Pclass ORDER BY Sex, Pclass LIMIT 50').df()

### Fitted LOGISTIC model on `titanic`
- Formula: `Survived ~ C(Sex) + C(Pclass) + Age + Q("Siblings/Spouses Aboard") + Q("Parents/Children Aboard") + Fare`
- Strongest predictor: `C(Sex)[T.male]` (negative effect, statistically significant at α=0.05, p=5.895e-43). A one-unit increase changes the odds by ~-93.6% (odds ratio = 0.064).
- pseudo-R² = 0.3397
- AIC = 796.93, BIC = 835.23

In [ ]:
import statsmodels.formula.api as smf
df = con.sql("SELECT * FROM titanic").df()
model = smf.logit("Survived ~ C(Sex) + C(Pclass) + Age + Q("Siblings/Spouses Aboard") + Q("Parents/Children Aboard") + Fare", data=df).fit(disp=0)
model.summary()

### Evaluated `survival_logit` on `titanic`
- ROC-AUC = 0.857, PR-AUC = 0.827, Brier = 0.139
- Accuracy = 0.803, F1 = 0.732 at threshold 0.5
- Calibration: 10 bins, max decile gap = 0.088

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, log_loss
y_true = titanic_df["Survived"]
y_pred = survival_logit.predict(titanic_df)
print(f"ROC-AUC = {roc_auc_score(y_true, y_pred):.3f}")
print(f"PR-AUC  = {average_precision_score(y_true, y_pred):.3f}")
print(f"Brier   = {brier_score_loss(y_true, y_pred):.3f}")
# Calibration table (quantile bins):
_q = pd.qcut(y_pred, q=10, duplicates='drop', labels=False)
calibration = pd.DataFrame({'bin': _q + 1, 'y_true': y_true, 'y_pred': y_pred}).groupby('bin').agg(
    mean_predicted=('y_pred', 'mean'), mean_observed=('y_true', 'mean'), n=('y_true', 'size')
).reset_index()
calibration

### Adjusted 7 p-values via Benjamini–Hochberg (α=0.05)
- 5 / 7 hypotheses rejected after correction
- Largest adjusted-p still significant: 0.0004054 (`Siblings/Spouses`)
- Not rejected: `Parents/Children`, `Fare`

In [ ]:
from statsmodels.stats.multitest import multipletests
p_raw = [5.894856816677544e-43, 0.0001137167998977293, 1.2247990676147229e-14, 2.5071128730764445e-08, 0.00028957608210445284, 0.36815134783533976, 0.25277116767477426]
labels = ['Sex[male]', 'Pclass[2]', 'Pclass[3]', 'Age', 'Siblings/Spouses', 'Parents/Children', 'Fare']
rejected, p_adj, _, _ = multipletests(p_raw, alpha=0.05, method='fdr_bh')
for lbl, raw, adj, rej in zip(labels, p_raw, p_adj, rejected):
    print(f"{lbl}  raw={raw:.4g}  adj={adj:.4g}  reject={rej}")